# Phase 2 - Preprocessing

The filtered_data/ files are already cleaned (artifacts removed, band-pass filtered 0.5-45 Hz). So our preprocessing is about preparing the data for feature extraction and classification:
    
In this Script:
1. Verify Frequency Content: Plot Power Spectral Density (PSD) to confirm the filtering and visualize how brain rhythms differ between stress and relaxation. This builds critical intuition for Phase 3.
    
2. Sub-Segmentation: Cut each 25-second trial into 5-second windows. This turns 480 trials into 2400 segments, giving us 5× more training samples. (The published research on this dataset uses 5s windows too.)
    
3. Normalization: Standardize signal amplitude per subject so the classifier focuses on brain patterns, not individual signal strength.
    
4. Save Preprocessed Data: Save the segmented, normalized data as a single .npz file for fast loading in Phase 3.

Prerequisites:
- Phase 1 completed (data loads, labels built)




In [1]:

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.io import loadmat
from scipy.signal import welch
from glob import glob
import warnings
warnings.filterwarnings('ignore')
 
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

## Configuration

In [2]:

# Accessing File paths from Notebooks/ directory

DATA_DIR = os.path.join("..", "Data", "SAM40")
OUTPUT_DIR = os.path.join("..", "Results", "phase2")
PHASE1_DIR = os.path.join("..", "Results", "phase1")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(DATA_DIR)
print(OUTPUT_DIR)
print(PHASE1_DIR)

..\Data\SAM40
..\Results\phase2
..\Results\phase1


## Constants

In [3]:


CHANNEL_NAMES = [
    'Cz', 'Fz', 'Fp1', 'F7', 'F3', 'FC1', 'C3', 'FC5',
    'FT9', 'T7', 'CP5', 'CP1', 'P3', 'P7', 'PO9', 'O1',
    'Pz', 'Oz', 'O2', 'PO10', 'P8', 'P4', 'CP2', 'CP6',
    'T8', 'FT10', 'FC6', 'C4', 'FC2', 'F4', 'F8', 'Fp2'
]
 
SAMPLING_RATE = 128
TRIAL_DURATION = 25       # in seconds
EXPECTED_SAMPLES = 3200   # 128 Hz × 25 s
N_CHANNELS = 32
N_SUBJECTS = 40
N_TRIALS = 3
 
# Segmentation parameters
WINDOW_SIZE_SEC = 5                         # Each window is 5 seconds
WINDOW_SIZE_SAMPLES = WINDOW_SIZE_SEC * SAMPLING_RATE  # 640 samples
WINDOWS_PER_TRIAL = TRIAL_DURATION // WINDOW_SIZE_SEC  # 5 windows per trial
 
TASK_NAMES = ['Relaxation', 'Stroop', 'Mirror_Image', 'Arithmetic']
TASK_COLORS = {
    'Relaxation': '#2ecc71', 'Stroop': '#e74c3c',
    'Mirror_Image': '#f39c12', 'Arithmetic': '#3498db',
}
 
# EEG frequency bands - Each band is associated with different mental states
EEG_BANDS = {
    'Delta':  (0.5,  4),   # Deep sleep - not very relevant for us
    'Theta':  (4,    8),   # Drowsiness, meditation, memory tasks
    'Alpha':  (8,   13),   # Relaxed wakefulness - DECREASES under stress
    'Beta':   (13,  30),   # Active thinking, focus - INCREASES under stress
    'Gamma':  (30,  45),   # High-level cognition, concentration
}
 
BAND_COLORS = {
    'Delta': '#3498db', 'Theta': '#2ecc71', 'Alpha': '#f1c40f',
    'Beta': '#e74c3c', 'Gamma': '#9b59b6',
}

## Data Loading

In [4]:

def load_single_mat(filepath):
    #Load one .mat file and return the EEG data as a numpy array
    
    mat_contents = loadmat(filepath)
    data_keys = [k for k in mat_contents.keys() if not k.startswith('__')]
    
    eeg_data = None
    for key in data_keys:
        arr = mat_contents[key]
        if isinstance(arr, np.ndarray):
            if arr.dtype == np.object_:
                try:
                    inner = arr.flat[0]
                    if hasattr(inner, 'dtype') and inner.dtype.names and 'data' in inner.dtype.names:
                        arr = inner['data'].flat[0]
                except:
                    continue
            if eeg_data is None or arr.size > eeg_data.size:
                eeg_data = arr.astype(np.float64)
    return eeg_data

In [5]:
def parse_filename(filename):
    #Extract subject, task, trial from filename (like Arithmetic_sub_1_trial1.mat)
    
    name = filename.replace('.mat', '')
    parts = name.split('_')
    subject_id = int(parts[parts.index('sub') + 1])
    trial_part = [p for p in parts if p.startswith('trial')][0]
    trial_num = int(trial_part.replace('trial', ''))
    
    fname_lower = filename.lower()
    if 'relax' in fname_lower:
        task_name = 'Relaxation'
    elif 'stroop' in fname_lower:
        task_name = 'Stroop'
    elif 'mirror' in fname_lower:
        task_name = 'Mirror_Image'
    elif 'arithmetic' in fname_lower:
        task_name = 'Arithmetic'
    else:
        task_name = 'Unknown'
    return subject_id, task_name, trial_num

In [6]:
def load_all_data(data_dir):
    # Load all filtered .mat files into a dictionary
    filtered_dir = os.path.join(data_dir, 'filtered_data')
    mat_files = sorted(glob(os.path.join(filtered_dir, '*.mat')))
    print(f"Found {len(mat_files)} .mat files")
    
    data_dict = {}
    for i, filepath in enumerate(mat_files):
        filename = os.path.basename(filepath)
        if (i + 1) % 100 == 0 or i == 0:
            print(f"  [{i+1}/{len(mat_files)}] {filename}")
        try:
            subject_id, task_name, trial_num = parse_filename(filename)
            eeg_data = load_single_mat(filepath)
            if eeg_data is not None:
                data_dict.setdefault(subject_id, {}).setdefault(task_name, {})[trial_num] = eeg_data
        except Exception as e:
            print(f"  ERROR: {filename} → {e}")
    
    total = sum(len(trials) for subj in data_dict.values() for trials in subj.values())
    print(f"Loaded {total} trials from {len(data_dict)} subjects")
    return data_dict

# Power Spectral Density (PSD) Analysis

Compute the Power Spectral Density of an EEG signal using Welch's method.

PSD shows how much "power" (energy) your signal has at each frequency. For EEG:
- If alpha (8-13 Hz) is high → the brain is in a relaxed state
- If beta (13-30 Hz) is high → the brain is actively processing
- Stress typically decreases alpha and increases beta

Welch's Method:
* We break the signal into overlapping segments, compute the FFT of each, and average them. This reduces noise in the estimate compared to a single FFT of the whole signal.

In [7]:

def compute_psd(signal, fs=SAMPLING_RATE):

    #Parameters:
    #     signal: 1D array of EEG samples
    #     fs: sampling rate (128 Hz)
    #Returns:
    #     freqs: array of frequencies (Hz)
    #     psd: power at each frequency (μV²/Hz)
    
    freqs, psd = welch(signal, fs=fs, nperseg=min(256, len(signal)), noverlap=128, scaling='density')
    # nperseg = number of samples per segment for Welch's method
    # Using 2 seconds (256 samples) gives good frequency resolution (~0.5 Hz)
    return freqs, psd

Computing the power in a specific frequency band. This sums up the PSD values between the band's lower and upper frequency limits.

In [8]:
def compute_band_power(freqs, psd, band_range):

    #Parameters:
    #     freqs: frequency array from compute_psd
    #     psd: power array from compute_psd
    #     band_range: tuple (low_freq, high_freq), e.g., (8, 13) for alpha
    
    #Returns:
    #     band_power: total power in that frequency range

    low, high = band_range
    # Find which frequency indices fall within the band
    idx = np.where((freqs >= low) & (freqs <= high))[0]
    # Integrate (sum) the power in that range
    # Multiply by frequency resolution for proper integration
    freq_res = freqs[1] - freqs[0]
    band_power = np.sum(psd[idx]) * freq_res
    return band_power

Comparing PSD across all 4 tasks to see how brain rhythms change under different cognitive loads.

What to look for:
- Alpha peak (~10 Hz) should be HIGHEST during Relaxation
- Beta activity (13-30 Hz) should be HIGHEST during stress tasks
- This is the fundamental signal that our classifier will learn

In [9]:
def plot_1_psd_comparison(data_dict, output_dir):

    # Average PSD across all subjects for each task
    print("\nPlot 1: PSD comparison across tasks")
    
    # Collect PSDs for Fz channel (frontal midline, good stress indicator)
    ch_idx = 1  # Fz
    task_psds = {task: [] for task in TASK_NAMES}
    
    for subject_id in sorted(data_dict.keys()):
        for task_name in TASK_NAMES:
            if task_name in data_dict[subject_id]:
                for trial_num in data_dict[subject_id][task_name]:
                    eeg = data_dict[subject_id][task_name][trial_num]
                    signal = eeg[ch_idx]
                    freqs, psd = compute_psd(signal)
                    task_psds[task_name].append(psd)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Power Spectral Density — Channel Fz (Frontal Midline)\n'
                 'Averaged across all subjects and trials',
                 fontsize=13, fontweight='bold')
    
    # --- Left panel: Full PSD curves ---
    ax = axes[0]
    for task_name in TASK_NAMES:
        mean_psd = np.mean(task_psds[task_name], axis=0)
        std_psd = np.std(task_psds[task_name], axis=0)
        ax.semilogy(freqs, mean_psd, color=TASK_COLORS[task_name],
                     linewidth=2, label=task_name.replace('_', ' '))
        ax.fill_between(freqs, mean_psd - std_psd/2, mean_psd + std_psd/2,
                        color=TASK_COLORS[task_name], alpha=0.15)
    
    # Shade the frequency bands
    band_alpha = 0.06
    for band_name, (lo, hi) in EEG_BANDS.items():
        ax.axvspan(lo, hi, alpha=band_alpha, color=BAND_COLORS[band_name])
        ax.text((lo + hi) / 2, ax.get_ylim()[1] * 0.7, band_name,
                ha='center', fontsize=8, color='gray', style='italic')
    
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Power Spectral Density (μV²/Hz)')
    ax.set_title('A. PSD curves by task', fontweight='bold')
    ax.set_xlim(0.5, 45)
    ax.legend(fontsize=9)
    
    # --- Right panel: Band power bar chart ---
    ax = axes[1]
    band_names = list(EEG_BANDS.keys())
    x = np.arange(len(band_names))
    width = 0.2
    
    for i, task_name in enumerate(TASK_NAMES):
        mean_psd_all = np.mean(task_psds[task_name], axis=0)
        powers = []
        for band_name in band_names:
            bp = compute_band_power(freqs, mean_psd_all, EEG_BANDS[band_name])
            powers.append(bp)
        ax.bar(x + i * width, powers, width, label=task_name.replace('_', ' '),
               color=TASK_COLORS[task_name], alpha=0.8, edgecolor='white')
    
    ax.set_xlabel('Frequency Band')
    ax.set_ylabel('Band Power (μV²)')
    ax.set_title('B. Band power by task', fontweight='bold')
    ax.set_xticks(x + 1.5 * width)
    ax.set_xticklabels(band_names)
    ax.legend(fontsize=9)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, '01_psd_comparison.png'),
                dpi=150, bbox_inches='tight')
    plt.close()
    print("  ✓ Saved: 01_psd_comparison.png")

Plotting the Beta/Alpha power ratio, a well-known stress indicator.
    
Research consistently shows that stress increases beta activity (active thinking) while decreasing alpha activity (relaxation). The ratio Beta/Alpha captures both changes in a single number:
- Low ratio → relaxed state (high alpha, low beta)
- High ratio → stressed state (low alpha, high beta)
    
This is one of the most commonly used features in EEG stress classification.

In [10]:

def plot_2_alpha_beta_ratio(data_dict, output_dir):
    print("\nPlot 2: Beta/Alpha ratio as a stress indicator")
    
    ratios_by_task = {task: [] for task in TASK_NAMES}
    
    # Compute Beta/Alpha ratio for every trial, averaged across all channels
    for subject_id in sorted(data_dict.keys()):
        for task_name in TASK_NAMES:
            if task_name in data_dict[subject_id]:
                for trial_num in data_dict[subject_id][task_name]:
                    eeg = data_dict[subject_id][task_name][trial_num]
                    
                    trial_ratios = []
                    for ch in range(N_CHANNELS):
                        freqs, psd = compute_psd(eeg[ch])
                        alpha_power = compute_band_power(freqs, psd, EEG_BANDS['Alpha'])
                        beta_power = compute_band_power(freqs, psd, EEG_BANDS['Beta'])
                        # Avoid division by zero
                        if alpha_power > 0:
                            trial_ratios.append(beta_power / alpha_power)
                    
                    if trial_ratios:
                        ratios_by_task[task_name].append(np.mean(trial_ratios))
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Beta/Alpha Power Ratio — A Classic Stress Indicator\n'
                 'Higher ratio = more stress (more beta, less alpha)',
                 fontsize=13, fontweight='bold')
    
    # --- Left: Box plot ---
    ax = axes[0]
    bp_data = [ratios_by_task[t] for t in TASK_NAMES]
    bp = ax.boxplot(bp_data, labels=[t.replace('_', '\n') for t in TASK_NAMES],
                    patch_artist=True, widths=0.6)
    for patch, task in zip(bp['boxes'], TASK_NAMES):
        patch.set_facecolor(TASK_COLORS[task])
        patch.set_alpha(0.5)
    ax.set_ylabel('Beta/Alpha Ratio')
    ax.set_title('A. Ratio distribution by task', fontweight='bold')
    
    # --- Right: Per-subject comparison (Relaxation vs Arithmetic) ---
    ax = axes[1]
    # Average ratio per subject for Relaxation and Arithmetic
    subj_relax = {}
    subj_arith = {}
    
    for subject_id in sorted(data_dict.keys()):
        for task_name in ['Relaxation', 'Arithmetic']:
            if task_name in data_dict[subject_id]:
                subj_ratios = []
                for trial_num in data_dict[subject_id][task_name]:
                    eeg = data_dict[subject_id][task_name][trial_num]
                    ch_ratios = []
                    for ch in range(N_CHANNELS):
                        freqs, psd = compute_psd(eeg[ch])
                        alpha_p = compute_band_power(freqs, psd, EEG_BANDS['Alpha'])
                        beta_p = compute_band_power(freqs, psd, EEG_BANDS['Beta'])
                        if alpha_p > 0:
                            ch_ratios.append(beta_p / alpha_p)
                    subj_ratios.append(np.mean(ch_ratios))
                
                if task_name == 'Relaxation':
                    subj_relax[subject_id] = np.mean(subj_ratios)
                else:
                    subj_arith[subject_id] = np.mean(subj_ratios)
    
    common_subjects = sorted(set(subj_relax.keys()) & set(subj_arith.keys()))
    relax_vals = [subj_relax[s] for s in common_subjects]
    arith_vals = [subj_arith[s] for s in common_subjects]
    
    x = np.arange(len(common_subjects))
    ax.bar(x - 0.2, relax_vals, 0.35, label='Relaxation', color='#2ecc71', alpha=0.7)
    ax.bar(x + 0.2, arith_vals, 0.35, label='Arithmetic', color='#3498db', alpha=0.7)
    ax.set_xlabel('Subject ID')
    ax.set_ylabel('Beta/Alpha Ratio')
    ax.set_title('B. Per-subject: Relaxation vs Arithmetic', fontweight='bold')
    ax.set_xticks(x[::5])
    ax.set_xticklabels([common_subjects[i] for i in range(0, len(common_subjects), 5)])
    ax.legend(fontsize=9)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, '02_beta_alpha_ratio.png'),
                dpi=150, bbox_inches='tight')
    plt.close()
    print("  ✓ Saved: 02_beta_alpha_ratio.png")

# Sub Segmentation

Cut each 25-second trial into 5-second non-overlapping windows and normalize per subject.

Why Sub Segment?
- 480 trials is a small dataset for ML. Cutting into 5s windows gives us 2400 samples - 5× more data for training.
- 5 seconds is long enough to capture several cycles of alpha/beta waves (alpha at 10 Hz has 50 full cycles in 5 seconds).
- The published research on SAM 40 also uses 5-second segments.
- Each window from the same trial inherits that trial's stress label.

Why Normalize per Subject?

Different people's brains produce signals at different amplitudes. Subject A might have signals ranging ±20 μV, while Subject B ranges ±60 μV. Without normalization, the classifier might learn "high amplitude = stressed" when really it just means "this is Subject B."

We z-score normalize each subject's data independently:
      `normalized = (signal - mean) / std`

This is computed per subject across all their trials, so the classifier learns from the signal PATTERNS, not absolute amplitudes.

NOTE: We normalize per subject BEFORE splitting into train/test sets in LOSO-CV. Since each subject's data is only used for either training or testing (never both), there's no data leakage.

In [11]:

def segment_and_normalize(data_dict, labels_df):
    print(f"\n{'='*60}")
    print(f"SUB-SEGMENTATION & NORMALIZATION")
    print(f"{'='*60}")
    print(f"Window size: {WINDOW_SIZE_SEC}s ({WINDOW_SIZE_SAMPLES} samples)")
    print(f"Windows per trial: {WINDOWS_PER_TRIAL}")
    
    # --- Step A: Compute per-subject statistics for normalization ---
    print(f"\nStep A: Computing per-subject normalization stats...")
    subject_stats = {}
    for subject_id in sorted(data_dict.keys()):
        # Collect ALL data from this subject to compute mean/std
        all_signals = []
        for task in data_dict[subject_id]:
            for trial in data_dict[subject_id][task]:
                all_signals.append(data_dict[subject_id][task][trial])
        
        # Stack all trials: shape (n_trials, 32, 3200) → compute stats
        all_data = np.concatenate(all_signals, axis=1)  # (32, n_trials×3200)
        
        # Compute mean and std per channel for this subject
        ch_mean = all_data.mean(axis=1, keepdims=True)  # (32, 1)
        ch_std = all_data.std(axis=1, keepdims=True)    # (32, 1)
        ch_std[ch_std < 1e-8] = 1.0  # Avoid division by zero
        
        subject_stats[subject_id] = (ch_mean, ch_std)
    
    print(f"  Computed stats for {len(subject_stats)} subjects")
    
    # --- Step B: Segment and normalize ---
    print(f"Step B: Segmenting trials into {WINDOW_SIZE_SEC}s windows...")
    
    segments = []
    seg_labels = []
    
    for subject_id in sorted(data_dict.keys()):
        ch_mean, ch_std = subject_stats[subject_id]
        
        for task_name in TASK_NAMES:
            if task_name not in data_dict[subject_id]:
                continue
            
            for trial_num in sorted(data_dict[subject_id][task_name].keys()):
                eeg = data_dict[subject_id][task_name][trial_num]
                
                # Normalize this trial using subject-level stats
                eeg_norm = (eeg - ch_mean) / ch_std
                
                # Cut into non-overlapping 5-second windows
                for win_idx in range(WINDOWS_PER_TRIAL):
                    start = win_idx * WINDOW_SIZE_SAMPLES
                    end = start + WINDOW_SIZE_SAMPLES
                    
                    if end <= eeg_norm.shape[1]:
                        segment = eeg_norm[:, start:end]  # (32, 640)
                        segments.append(segment)
                        
                        # Find this trial's stress label
                        label_row = labels_df[
                            (labels_df['subject'] == subject_id) &
                            (labels_df['task'] == task_name) &
                            (labels_df['trial'] == trial_num)
                        ]
                        
                        if len(label_row) > 0:
                            rating = label_row.iloc[0]['rating']
                            stress_level = label_row.iloc[0]['stress_level']
                        else:
                            rating = 0
                            stress_level = 'Relaxed' if task_name == 'Relaxation' else 'Unknown'
                        
                        seg_labels.append({
                            'subject': subject_id,
                            'task': task_name,
                            'trial': trial_num,
                            'window': win_idx + 1,
                            'rating': rating,
                            'stress_level': stress_level,
                        })
    
    segments = np.array(segments)
    seg_labels_df = pd.DataFrame(seg_labels)
    
    # --- Summary ---
    print(f"\n{'='*60}")
    print(f"SEGMENTATION COMPLETE")
    print(f"{'='*60}")
    print(f"Original trials:    {sum(len(t) for s in data_dict.values() for t in s.values())}")
    print(f"Segments created:   {len(segments)}")
    print(f"Segment shape:      {segments[0].shape} (channels × samples)")
    print(f"Duration per segment: {WINDOW_SIZE_SEC}s")
    print(f"\nAmplitude after normalization:")
    print(f"  Mean: {segments.mean():.4f} (should be ~0)")
    print(f"  Std:  {segments.std():.4f} (should be ~1)")
    
    print(f"\nSegments per stress level:")
    for level in ['Relaxed', 'Low Stress', 'High Stress']:
        count = (seg_labels_df['stress_level'] == level).sum()
        print(f"  {level:12s}: {count:5d} ({count/len(seg_labels_df)*100:.1f}%)")
    
    return segments, seg_labels_df

In [12]:


def plot_3_segmentation_example(data_dict, output_dir):
    #Visualize how one 25-second trial gets cut into five 5-second segments.
    
    print("\nPlot 3: Segmentation visualization")
    
    # Get one trial
    subject_id = min(data_dict.keys())
    task = 'Arithmetic'
    if task not in data_dict[subject_id]:
        task = list(data_dict[subject_id].keys())[0]
    trial_num = min(data_dict[subject_id][task].keys())
    eeg = data_dict[subject_id][task][trial_num]
    
    ch_idx = 1  # Fz
    signal = eeg[ch_idx]
    time = np.arange(len(signal)) / SAMPLING_RATE
    
    fig, ax = plt.subplots(figsize=(16, 4))
    ax.plot(time, signal, color='#3498db', linewidth=0.6, alpha=0.8)
    
    # Shade each 5-second window with alternating colors
    colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']
    for i in range(WINDOWS_PER_TRIAL):
        t_start = i * WINDOW_SIZE_SEC
        t_end = t_start + WINDOW_SIZE_SEC
        ax.axvspan(t_start, t_end, alpha=0.1, color=colors[i])
        ax.axvline(t_start, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
        ax.text(t_start + WINDOW_SIZE_SEC/2, ax.get_ylim()[1] * 0.9,
                f'Seg {i+1}', ha='center', fontsize=10, fontweight='bold',
                color=colors[i])
    ax.axvline(TRIAL_DURATION, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
    
    ax.set_xlabel('Time (seconds)')
    ax.set_ylabel('Amplitude (μV)')
    ax.set_title(f'Segmentation: One 25s trial → Five 5s segments\n'
                 f'Subject {subject_id}, {task}, Trial {trial_num}, Channel Fz',
                 fontweight='bold')
    ax.set_xlim(0, TRIAL_DURATION)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, '03_segmentation_example.png'),
                dpi=150, bbox_inches='tight')
    plt.close()
    print("  ✓ Saved: 03_segmentation_example.png")

In [13]:

def plot_4_normalization_effect(data_dict, output_dir):

    """
    Show the effect of per-subject normalization on signal amplitudes.
    Before: different subjects have different amplitude ranges.
    After: all subjects are on the same scale (mean=0, std=1).
    """
    
    print("\nPlot 4: Normalization effect")
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Per-Subject Normalization Effect\n'
                 'Z-score normalization makes signal amplitudes comparable across subjects',
                 fontsize=13, fontweight='bold')
    
    ch_idx = 1  # Fz
    
    # Pick 5 subjects to show
    subjects_to_show = sorted(data_dict.keys())[:5]
    colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']
    
    for ax_idx, (ax, title) in enumerate(zip(axes, ['A. Before normalization', 'B. After normalization'])):
        for i, sid in enumerate(subjects_to_show):
            # Get one relaxation trial from this subject
            if 'Relaxation' in data_dict[sid]:
                trial_num = min(data_dict[sid]['Relaxation'].keys())
                signal = data_dict[sid]['Relaxation'][trial_num][ch_idx].copy()
                
                if ax_idx == 1:
                    # Normalize using this subject's stats
                    all_data = []
                    for task in data_dict[sid]:
                        for trial in data_dict[sid][task]:
                            all_data.append(data_dict[sid][task][trial][ch_idx])
                    all_concat = np.concatenate(all_data)
                    signal = (signal - all_concat.mean()) / (all_concat.std() + 1e-8)
                
                # Plot first 2 seconds
                time = np.arange(256) / SAMPLING_RATE
                ax.plot(time, signal[:256] + i * (15 if ax_idx == 0 else 5),
                        color=colors[i], linewidth=0.8, label=f'Subject {sid}')
        
        ax.set_xlabel('Time (seconds)')
        ax.set_ylabel('Amplitude' + (' (μV)' if ax_idx == 0 else ' (z-scored)'))
        ax.set_title(title, fontweight='bold')
        ax.legend(fontsize=8, loc='upper right')
        ax.set_xlim(0, 2)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, '04_normalization_effect.png'),
                dpi=150, bbox_inches='tight')
    plt.close()
    print("  ✓ Saved: 04_normalization_effect.png")

# MAIN

In [14]:
if __name__ == "__main__":
    print("=" * 60)
    print("  EEG COGNITIVE STRESS CLASSIFICATION")
    print("  Phase 2: Preprocessing")
    print("=" * 60)

  EEG COGNITIVE STRESS CLASSIFICATION
  Phase 2: Preprocessing


In [15]:
# Loading data

print(f"\n▶ Loading data from: {DATA_DIR}")
data_dict = load_all_data(DATA_DIR)
    
if not data_dict:
    print("\n⚠ Could not load data. Check DATA_DIR path.")
    exit(1)
    
# Load labels from Phase 1
labels_path = os.path.join(PHASE1_DIR, 'stress_labels.csv')
if os.path.exists(labels_path):
    labels_df = pd.read_csv(labels_path)
    print(f"Loaded labels: {len(labels_df)} rows")
else:
    print(f"⚠ Labels file not found at {labels_path}")
    print(f"  Run Phase 1 first to generate stress_labels.csv")
    exit(1)


▶ Loading data from: ..\Data\SAM40
Found 480 .mat files
  [1/480] Arithmetic_sub_10_trial1.mat
  [100/480] Arithmetic_sub_40_trial1.mat
  [200/480] Mirror_image_sub_34_trial2.mat
  [300/480] Relax_sub_28_trial3.mat
  [400/480] Stroop_sub_22_trial1.mat
Loaded 480 trials from 40 subjects
Loaded labels: 480 rows


In [16]:
# PSD Analysis (frequency content verification)

print(f"\n▶ Analyzing frequency content (PSD)...")
plot_1_psd_comparison(data_dict, OUTPUT_DIR)
plot_2_alpha_beta_ratio(data_dict, OUTPUT_DIR)


▶ Analyzing frequency content (PSD)...

Plot 1: PSD comparison across tasks
  ✓ Saved: 01_psd_comparison.png

Plot 2: Beta/Alpha ratio as a stress indicator
  ✓ Saved: 02_beta_alpha_ratio.png


In [17]:
# Segmentation and normalization

print(f"\n▶ Segmenting and normalizing...")
plot_3_segmentation_example(data_dict, OUTPUT_DIR)
plot_4_normalization_effect(data_dict, OUTPUT_DIR)
    
segments, seg_labels_df = segment_and_normalize(data_dict, labels_df)


▶ Segmenting and normalizing...

Plot 3: Segmentation visualization
  ✓ Saved: 03_segmentation_example.png

Plot 4: Normalization effect
  ✓ Saved: 04_normalization_effect.png

SUB-SEGMENTATION & NORMALIZATION
Window size: 5s (640 samples)
Windows per trial: 5

Step A: Computing per-subject normalization stats...
  Computed stats for 40 subjects
Step B: Segmenting trials into 5s windows...

SEGMENTATION COMPLETE
Original trials:    480
Segments created:   2400
Segment shape:      (32, 640) (channels × samples)
Duration per segment: 5s

Amplitude after normalization:
  Mean: 0.0000 (should be ~0)
  Std:  1.0000 (should be ~1)

Segments per stress level:
  Relaxed     :  1110 (46.2%)
  Low Stress  :   940 (39.2%)
  High Stress :   350 (14.6%)


In [18]:
# Save Preprocessed data

print(f"\n▶ Saving preprocessed data...")
    
# Save as .npz (compressed numpy format - faster to load later)
save_path = os.path.join(OUTPUT_DIR, 'preprocessed_segments.npz')
np.savez_compressed(save_path, segments=segments)
    
# Save segment labels
labels_save_path = os.path.join(OUTPUT_DIR, 'segment_labels.csv')
seg_labels_df.to_csv(labels_save_path, index=False)
    
file_size_mb = os.path.getsize(save_path) / (1024 * 1024)
print(f"  ✓ Segments saved: {save_path} ({file_size_mb:.1f} MB)")
print(f"  ✓ Labels saved:   {labels_save_path}")


▶ Saving preprocessed data...
  ✓ Segments saved: ..\Results\phase2\preprocessed_segments.npz (360.1 MB)
  ✓ Labels saved:   ..\Results\phase2\segment_labels.csv


# Summary

In [19]:

print(f"\n{'='*60}")
print(f"  PHASE 2 COMPLETE!")
print(f"{'='*60}")
print(f"  Total segments:     {len(segments)}")
print(f"  Segment shape:      {segments[0].shape} ({N_CHANNELS} ch × {WINDOW_SIZE_SAMPLES} samples)")
print(f"  Window duration:    {WINDOW_SIZE_SEC} seconds")
print(f"  Normalized:         Yes (z-score per subject)")
print(f"\n  Outputs in: {os.path.abspath(OUTPUT_DIR)}/")
print(f"    01_psd_comparison.png       — Frequency analysis")
print(f"    02_beta_alpha_ratio.png     — Stress indicator")
print(f"    03_segmentation_example.png — Window visualization")
print(f"    04_normalization_effect.png  — Before/after normalization")
print(f"    preprocessed_segments.npz   — Segmented EEG data")
print(f"    segment_labels.csv          — Labels for each segment")
print(f"\n  Next → Phase 3: Feature Extraction")
print(f"{'='*60}")


  PHASE 2 COMPLETE!
  Total segments:     2400
  Segment shape:      (32, 640) (32 ch × 640 samples)
  Window duration:    5 seconds
  Normalized:         Yes (z-score per subject)

  Outputs in: c:\Users\hibro\OneDrive\Desktop\Desktop_Files\Projects\Python\ML_Models\Cognitive_Stress_Classification\EEG-Stress-Classification\Results\phase2/
    01_psd_comparison.png       — Frequency analysis
    02_beta_alpha_ratio.png     — Stress indicator
    03_segmentation_example.png — Window visualization
    04_normalization_effect.png  — Before/after normalization
    preprocessed_segments.npz   — Segmented EEG data
    segment_labels.csv          — Labels for each segment

  Next → Phase 3: Feature Extraction
